In [1]:
!pip install -qq torch \
    transformers \
    accelerate \
    bitsandbytes \
    huggingface-hub \
    sentence-transformers \
    pypdf \
    beautifulsoup4 \
    faiss-cpu \
    rapidocr \
    unstructured

!pip install -qq langchain langgraph langchain-core langchain-community langchain-openai \
    langchain-huggingface \
    langserve[all] \
    chromadb \
    langchain-chroma \
    wget \
    langchainhub \
    hf_xet \
    langchain-google-genai \
    langchain-qdrant \
    google-generativeai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 24.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 10.5 MB/s eta 0:00:00

In [2]:
import os
import re
import time
import glob
import wget
import torch
import multiprocessing
import warnings

from tqdm import tqdm
from typing import Union, List, Literal
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore")

In [3]:
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, chain
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_google_genai import ChatGoogleGenerativeAI

In [4]:
PDF_DIR = "data_source"
file_links = [
    {"title": "Attention Is All You Need", "url": "https://arxiv.org/pdf/1706.03762v7"},
    {
        "title": "LoRA: Low-Rank Adaptation of Large Language Models",
        "url": "https://arxiv.org/pdf/2106.09685v2",
    },
    {
        "title": "FedPara: Low-Rank Hadamard Product for Communication-Efficient Federated Learning",
        "url": "https://arxiv.org/pdf/2108.06098v3",
    },
    {
        "title": "Parameter-Efficient Orthogonal Finetuning via Butterfly Factorization",
        "url": "https://arxiv.org/pdf/2311.06243v2",
    },
    {
        "title": "Adaptive Budget Allocation for Parameter-Efficient Fine-Tuning",
        "url": "https://arxiv.org/pdf/2303.10512v2",
    },
    {
        "title": "LLaMA-Adapter: Efficient Fine-tuning of Language Models with Zero-init Attention",
        "url": "https://arxiv.org/pdf/2303.16199v3",
    },
    {
        "title": "Bone: Block Affine Transformation as Parameter Efficient Fine-tuning Methods for Large Language Models",
        "url": "https://arxiv.org/pdf/2409.15371v9",
    },
    {
        "title": "Bridging The Gap between Low-rank and Orthogonal Adaptation via Householder Reflection Adaptation",
        "url": "https://arxiv.org/pdf/2405.17484v3",
    },
    {
        "title": "Parameter-Efficient Fine-Tuning for Large Models: A Comprehensive Survey",
        "url": "https://arxiv.org/pdf/2403.14608v7",
    },
]

Download all required documents

In [5]:
os.makedirs(PDF_DIR, exist_ok=True)
for file_link in file_links:
    print(f'Downloading: {file_link["title"]}')
    if not os.path.exists(os.path.join(PDF_DIR, f"{file_link['title']}.pdf")):
        wget.download(file_link["url"], out=os.path.join(PDF_DIR, f"{file_link['title']}.pdf"))
        print(f"Downloaded: {file_link['title']}")
        time.sleep(1)
    else:
        print(f"File already exists: {file_link['title']}")
    print("#" * 50)

Downloading: Attention Is All You Need
Downloaded: Attention Is All You Need
##################################################
Downloading: LoRA: Low-Rank Adaptation of Large Language Models
Downloaded: LoRA: Low-Rank Adaptation of Large Language Models
##################################################
Downloading: FedPara: Low-Rank Hadamard Product for Communication-Efficient Federated Learning
Downloaded: FedPara: Low-Rank Hadamard Product for Communication-Efficient Federated Learning
##################################################
Downloading: Parameter-Efficient Orthogonal Finetuning via Butterfly Factorization
Downloaded: Parameter-Efficient Orthogonal Finetuning via Butterfly Factorization
##################################################
Downloading: Adaptive Budget Allocation for Parameter-Efficient Fine-Tuning
Downloaded: Adaptive Budget Allocation for Parameter-Efficient Fine-Tuning
##################################################
Downloading: LLaMA-Adapter: Efficien

In [6]:
def load_pdf(pdf_file):
    """Load and clean a PDF file"""
    docs = PyPDFLoader(pdf_file, extract_images=True).load()
    for doc in docs:
        doc.page_content = "".join(char for char in doc.page_content if ord(char) < 128)
    return docs


doc_loaded = []
pbar = tqdm(sorted(os.listdir(PDF_DIR)))
for filename in pbar:
    pbar.set_postfix_str(f"Processing \"{filename}\"")
    file = os.path.join(PDF_DIR, filename)
    if not filename.endswith(".pdf"):
        continue
    if os.path.getsize(file) == 0:
        print(f"File is empty: {file}")
        continue
    doc_loaded.extend(load_pdf(file))

 11%|█         | 1/9 [00:00<00:06,  1.22it/s, Processing "Attention Is All You Need.pdf"]                                     [INFO] 2026-05-31 13:27:22,520 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-31 13:27:22,558 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-31 13:27:23,436 [RapidOCR] download_file.py:82: Download size: 4.53MB
[INFO] 2026-05-31 13:27:23,529 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-31 13:27:23,532 [RapidOCR] main.py:57: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-31 13:27:23,745 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-31 13:27:23,748 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/

Split documents into small chunks of text using ```RecursiveCharacterTextSplitte```. You have to split the documents by using the ```"\n\n", "\n", " ", ""``` separators. Each text chunk contains around ```1024``` characters, and the number of overlapping characters between two chunks is ```128``` chars.

Complete the following code piece with the above requirements.

HINT: You have to use ```separators```, ```chunk_size```, and ```chunk_overlap``` arguments.

In [7]:
### TODO: fill in the following ... to create the text splitter
# HINT: use separators, chunk_size, and chunk_overlap arguments.
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],  # Fill in the separators argument
    chunk_size=1024,  # Fill in the chunk_size argument
    chunk_overlap=128   # Fill in the chunk_overlap argument
)
### END


text_chunks = text_splitter.split_documents(doc_loaded)
print(f"Split all pdf documents into {len(text_chunks)} text chunks.") # Use this output to answer the question


Split all pdf documents into 994 text chunks.


In [8]:
# Build Vector Store Database and get Retriever
device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

vector_store = Chroma(collection_name="pdf", embedding_function=embeddings, persist_directory="./db")
vector_store.add_documents(text_chunks)

retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 1})


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Use retriever to search for paragraphs that are related to the following phrase: `background and mathematics of low-rank adaptation`

HINT: use `invoke()` method

In [9]:
### TODO: search for paragraphs related to the following phrase: background and mathematics of low-rank adaptation
# HINT: use invoke() method
results = retriever.invoke("background and mathematics of low-rank adaptation")
### END

print(results[0].page_content)


Low Rank Adaptation. LoRA (Hu et al., 2022) models the incremental update of the pre-trained
weights by the product of two small matrices. For h = W (0)x, the modified forward pass is:
h = W (0)x + x = W (0)x + BAx, (2)
where W (0),   Rd1d2, A  Rrd2 and B  Rd1r with r  { d1, d2}. A typically adopts a
random Gaussion initialization while B is initialized with zero to have  = 0 at the beginning of
3


Create the LLM with structured output, containing the short answer and the explanation.

Use `gemini-2.5-flash-lite` model with `temperature=0.0` and `top_k=1` for consistent outputs.

Remember to get the Google Gemini API key at [https://aistudio.google.com/apikey](https://aistudio.google.com/apikey).

In Google Colab, add your key to **Secrets** with the name `GOOGLE_API_KEY`, then enable access for this notebook.


In [10]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

### TODO: Initialize our LLM
# HINT: use model, temperature, and top_k arguments.
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.0,
    top_k=1,
    google_api_key=GOOGLE_API_KEY,
    max_retries=2,
    timeout=30,
    max_tokens=1024,
)
### END

class OutputWithExplanation(BaseModel):
    answer: str = Field(description="The short answer to the question.")
    explanation: str = Field(
        description="The detailed step-by-step explanation for the answer based on the provided context."
    )

structured_llm = llm.with_structured_output(OutputWithExplanation)


Build prompt template


In [11]:
@chain
def get_input_prompt(inputs):
    prompt_template = ChatPromptTemplate.from_template("""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}

The response must contain the short answer and a detailed explanation, which is described in the following schema:\n"""
)
    res = prompt_template.invoke({"context": inputs["context"], "question": inputs["question"]})
    res.messages[0].content = res.messages[0].content + f"{OutputWithExplanation.model_json_schema()}"
    return res

def get_context_from_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

Build RAG chain

In [12]:
### TODO: build RAG chain.
# HINT: use | between each element to construct the chain
input_data = {"context": retriever | get_context_from_docs, "question": RunnablePassthrough()}
rag_chain = input_data | get_input_prompt | structured_llm
### END


Run the RAG chain with query: `What is the mathematical background of low-rank adaptation?`

In [13]:
### TODO: run RAG chain with query: What is the mathematical background of low-rank adaptation?
rag_output = rag_chain.invoke("What is the mathematical background of low-rank adaptation?")
### END

print(f'Answer: {rag_output.answer}')
print(f'Explanation: {rag_output.explanation}')


Answer: Low-Rank Adaptation (LoRA) models incremental updates to pre-trained weights using the product of two small matrices.
Explanation: In LoRA, the modified forward pass is represented as h = W(0)x + BAx. Here, W(0) represents the original pre-trained weights, and BA represents the low-rank update, where B and A are smaller matrices. The matrix A is typically initialized with random Gaussian values, while B is initialized with zeros, ensuring that the initial update (BA) is zero.
